# Notebook de dev pour l'entrainement vers MLflow

In [1]:
import logging
import numpy as np

import os
import mlflow
from mlflow import log_metric, log_param, log_artifacts
from random import random, randint
from datetime import datetime
import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
import time

from scipy.stats.mstats import winsorize

import pandas as pd
import numpy as np

import boto3


from dotenv import load_dotenv
load_dotenv(override=True)


/Users/manjakaranjatoson/anaconda3/envs/env_ppml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [99]:

# MLFLOW_TRACKING_URI = os.environ["MLFLOW_TRACKING_URI"]

In [100]:
# MLFLOW_TRACKING_URI

In [101]:


# # Set tracking URI to your Hugging Face application
# mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# if __name__ == "__main__":
#     # Log a parameter (key-value pair)
#     log_param("param1", randint(0, 100))

#     # Log a metric; metrics can be updated throughout the run
#     log_metric("foo", random())
#     log_metric("foo", random() + 1)
#     log_metric("foo", random() + 2)

#     # Log an artifact (output file)
#     if not os.path.exists("outputs"):
#         os.makedirs("outputs")
#     with open("outputs/test.txt", "w") as f:
#         f.write("hello world!")
#     log_artifacts("outputs")


# 1. Chargement des données

### Option 1. Via SQL

### Option 2. Via CSV

In [3]:
from dotenv import load_dotenv, find_dotenv
import os
import pandas as pd

load_dotenv(find_dotenv(), override=True)

def get_s3_parquet(s3_key):
    bucket = os.getenv("BUCKET_EQUIPE")
    region = os.getenv("AWS_DEFAULT_REGION_EQUIPE", "eu-north-1")
    access_key = os.getenv("AWS_ACCESS_KEY_ID_EQUIPE")
    secret_key = os.getenv("AWS_SECRET_ACCESS_KEY_EQUIPE")

    print("bucket =", bucket)

    df = pd.read_parquet(
        f"s3://{bucket}/{s3_key}",
        storage_options={
            "key": access_key,
            "secret": secret_key,
            "client_kwargs": {"region_name": region},
        },
    )
    return df

In [12]:

# df_ppml = get_s3_parquet("prod/train/full_training_2025-10-05_to_2026-04-06.parquet")

df_ppml = pd.read_parquet("/Users/manjakaranjatoson/Desktop/A_I/JEDHA/PROJECTS/CDSD/PPML/data/processed/test_patrick/full_training_2025-10-05_to_2026-04-06.parquet")
print(df_ppml.shape)
df_ppml.head()
df_ppml.columns

(264759, 48)


Index(['date', 'icao', 'type', 'flight_number', 'call_sign', 'status',
       'codeshare_status', 'is_cargo', 'scheduled_utc', 'revised_utc',
       'runway_utc', 'delay_minutes', 'terminal_dep', 'terminal_arr',
       'gate_dep', 'baggage_belt', 'destination_icao', 'destination_iata',
       'destination_name', 'airline_name', 'airline_iata', 'airline_icao',
       'aircraft_model', 'aircraft_reg', 'aircraft_mode_s', 'quality_dep',
       'quality_arr', 'aircraft_family', 'num_seats', 'is_widebody',
       'is_narrowbody', 'is_regional', 'aircraft_size_category',
       'is_freighter', 'is_holiday', 'vac_school', 'is_holiday_eve',
       'is_holiday_next', 'is_weekend', 'is_weekend_or_holiday',
       'holiday_name', 'temperature_2m', 'relative_humidity_2m',
       'wind_speed_10m', 'wind_gusts_10m', 'pressure_msl', 'precipitation',
       'cloud_cover'],
      dtype='object')

### colonnes aerodatabox 
'date', 'icao', 'type', 'flight_number', 'call_sign', 'status',
       'codeshare_status', 'is_cargo', 'scheduled_utc', 'revised_utc',
       'runway_utc', 'delay_minutes', 'terminal_dep', 'terminal_arr',
       'gate_dep', 'baggage_belt', 'destination_icao', 'destination_iata',
       'destination_name', 'airline_name', 'airline_iata', 'airline_icao',
       'aircraft_model', 'aircraft_reg', 'aircraft_mode_s', 'quality_dep',
       'quality_arr', 
### colonnes features appareil :
'aircraft_family', 'num_seats', 'is_widebody',
       'is_narrowbody', 'is_regional', 'aircraft_size_category',
       'is_freighter'
### colonnes feature ferié(librairie python holiday)
'is_holiday', 'is_holiday_eve', 'is_holiday_next',
       'is_weekend', 'is_weekend_or_holiday', 'holiday_name', 'temperature_2m',
       'relative_humidity_2m', 'wind_speed_10m', 'wind_gusts_10m',
       'pressure_msl', 'precipitation', 'cloud_cover'
### TODO VACANCES SCOLAIRES ! 

In [11]:
# df_predict = get_s3_parquet("prod/predict/prediction_avril2026.parquet")

# df_predict.columns

df_predict = pd.read_parquet("/Users/manjakaranjatoson/Desktop/A_I/JEDHA/PROJECTS/CDSD/PPML/data/processed/test_patrick/prediction_avril2026.parquet")
print(df_predict.shape)
df_predict.columns

(35604, 48)


Index(['date', 'icao', 'type', 'flight_number', 'call_sign', 'status',
       'codeshare_status', 'is_cargo', 'scheduled_utc', 'revised_utc',
       'runway_utc', 'delay_minutes', 'terminal_dep', 'terminal_arr',
       'gate_dep', 'baggage_belt', 'destination_icao', 'destination_iata',
       'destination_name', 'airline_name', 'airline_iata', 'airline_icao',
       'aircraft_model', 'aircraft_reg', 'aircraft_mode_s', 'quality_dep',
       'quality_arr', 'aircraft_family', 'num_seats', 'is_widebody',
       'is_narrowbody', 'is_regional', 'aircraft_size_category',
       'is_freighter', 'is_holiday', 'vac_school', 'is_holiday_eve',
       'is_holiday_next', 'is_weekend', 'is_weekend_or_holiday',
       'holiday_name', 'temperature_2m', 'relative_humidity_2m',
       'wind_speed_10m', 'wind_gusts_10m', 'pressure_msl', 'precipitation',
       'cloud_cover'],
      dtype='object')

***

# 2. EDA initiale

### 1. Quelques stats

In [105]:
df_ppml.shape

(264759, 48)

In [106]:
df_ppml.describe()


,scheduled_utc,delay_minutes,num_seats,is_holiday,vac_school,is_holiday_eve,is_holiday_next,is_weekend,is_weekend_or_holiday,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_gusts_10m,pressure_msl,precipitation,cloud_cover
count,264759,250295.000000,264759.000000,264759.000000,264759.000000,264759.000000,264759.000000,264759.000000,264759.000000,264759.000000,264759.000000,264759.000000,264759.000000,264759.000000,264759.000000,264759.000000
mean,2026-01-21 06:12:46.287151616,15.806756,175.783535,0.041744,0.418947,0.038197,0.016793,0.247716,0.288957,8.320498,80.862675,12.969064,27.613549,1016.292819,0.075689,69.659660
min,2025-10-04 20:05:00,-695.000000,48.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-5.900000,25.000000,0.000000,1.100000,979.800000,0.000000,0.000000
25%,2025-12-18 17:25:00,-3.000000,135.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,72.000000,8.700000,18.700000,1011.400000,0.000000,33.000000
50%,2026-01-17 15:20:00,12.000000,165.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.900000,84.000000,12.100000,25.900000,1018.500000,0.000000,98.000000
75%,2026-03-03 06:35:00,26.000000,165.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,11.600000,92.000000,17.100000,35.600000,1022.500000,0.000000,100.000000
max,2026-04-06 10:05:00,1233.000000,525.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,23.700000,100.000000,38.700000,83.900000,1034.200000,9.300000,100.000000
std,NaN,32.446300,76.519658,0.200003,0.493388,0.191672,0.128494,0.431687,0.453279,4.803816,13.467027,5.909180,11.817885,8.942830,0.292108,39.745377


In [107]:
df_ppml.head()

,date,icao,type,flight_number,call_sign,status,codeshare_status,is_cargo,scheduled_utc,revised_utc,runway_utc,delay_minutes,terminal_dep,terminal_arr,gate_dep,baggage_belt,destination_icao,destination_iata,destination_name,airline_name,airline_iata,airline_icao,aircraft_model,aircraft_reg,aircraft_mode_s,quality_dep,quality_arr,aircraft_family,num_seats,is_widebody,is_narrowbody,is_regional,aircraft_size_category,is_freighter,is_holiday,vac_school,is_holiday_eve,is_holiday_next,is_weekend,is_weekend_or_holiday,holiday_name,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_gusts_10m,pressure_msl,precipitation,cloud_cover
0,2025-10-05,LFPG,departure,BJ 521,LBT521,Departed,IsOperator,False,2025-10-04 20:05:00,2025-10-04 23:28Z,2025-10-04 23:28Z,203.0,3,None,None,None,DTTA,TUN,Tunis,Nouvelair Tunisie,BJ,LBT,Airbus A320,TS-INM,02A1AC,"[Basic, Live]",[],Airbus A320 Family,165,False,True,False,Medium,False,0,0,0,0,1,1,None,10.6,75,19.6,41.4,1013.8,0.0,23
1,2025-10-05,LFPG,departure,BJ 543,LBT543,Departed,IsOperator,False,2025-10-04 20:15:00,2025-10-04 22:35Z,2025-10-04 22:35Z,140.0,3,None,None,02,DTMB,MIR,Monastir,Nouvelair Tunisie,BJ,LBT,Airbus A320,TS-INR,02A1B1,"[Basic, Live]","[Basic, Live]",Airbus A320 Family,165,False,True,False,Medium,False,0,0,0,0,1,1,None,10.6,75,19.6,41.4,1013.8,0.0,23
2,2025-10-05,LFPG,departure,AF 406,AFR406,Departed,IsOperator,False,2025-10-04 21:20:00,2025-10-04 22:14Z,2025-10-04 22:14Z,54.0,2E,2,None,None,SCEL,SCL,Santiago,Air France,AF,AFR,Airbus A350-900,F-HTYT,39CF13,"[Basic, Live]",[Basic],Airbus A350,325,True,False,False,Very Large,False,0,0,0,0,1,1,None,10.4,77,19.9,38.9,1014.2,0.0,15
3,2025-10-05,LFPG,departure,AF 454,AFR454,Departed,IsOperator,False,2025-10-04 21:30:00,2025-10-04 22:03Z,2025-10-04 22:03Z,33.0,2E,3,None,None,SBGR,GRU,São Paulo,Air France,AF,AFR,Boeing 777-300,F-GSQY,394A18,"[Basic, Live]","[Basic, Live]",Boeing 777,355,True,False,False,Very Large,False,0,0,0,0,1,1,None,10.4,77,19.9,38.9,1014.2,0.0,15
4,2025-10-05,LFPG,departure,AF 116,AFR116,Departed,IsOperator,False,2025-10-04 21:45:00,2025-10-04 22:04Z,2025-10-04 22:04Z,19.0,2E,1,None,13,ZSPD,PVG,Shanghai,Air France,AF,AFR,Boeing 777,F-GZNS,3965B2,"[Basic, Live]","[Basic, Live]",Boeing 777,355,True,False,False,Very Large,False,0,0,0,0,1,1,None,10.1,77,19.9,38.9,1014.4,0.0,20


# Colonnes autorisées pour l'entraînement ET pour la prédiction
features_autorisees = [
    'icao', 'type', 'flight_number', 'codeshare_status', 'is_cargo',
    'scheduled_utc', 'terminal_dep', 'terminal_arr',
    'airline_name', 'aircraft_model', 'aircraft_family', 'aircraft_size_category',
    'num_seats', 'is_widebody', 'is_narrowbody', 'is_regional', 'is_freighter',
    'is_holiday', 'vac_school', 'is_holiday_eve', 'is_holiday_next', 'is_weekend',
    'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'wind_gusts_10m',
    'pressure_msl', 'precipitation', 'cloud_cover',
    'dest_icao_clean', 'dep_hour', 'dep_dayofweek', 'dep_month',
    'dep_hour_sin', 'dep_hour_cos', 'dep_dayofweek_sin', 'dep_dayofweek_cos', 
    'period_of_day'
]

# Colonnes a ABSOLUMENT supprimer du train (car leakage)
cols_leakage = ['status', 'delay_minutes', 'revised_utc', 'runway_utc', 
                'gate_dep', 'baggage_belt', 'quality_dep', 'quality_arr']

### Colonnes vides dans le predict:


In [13]:
df_predict.dtypes

date                              object
icao                              object
type                              object
flight_number                     object
call_sign                         object
status                            object
codeshare_status                  object
is_cargo                            bool
scheduled_utc             datetime64[ns]
revised_utc                       object
runway_utc                        object
delay_minutes                    float64
terminal_dep                      object
terminal_arr                      object
gate_dep                          object
baggage_belt                      object
destination_icao                  object
destination_iata                  object
destination_name                  object
airline_name                      object
airline_iata                      object
airline_icao                      object
aircraft_model                    object
aircraft_reg                      object
aircraft_mode_s 

In [109]:
df_predict.head()

,date,icao,type,flight_number,call_sign,status,codeshare_status,is_cargo,scheduled_utc,revised_utc,runway_utc,delay_minutes,terminal_dep,terminal_arr,gate_dep,baggage_belt,destination_icao,destination_iata,destination_name,airline_name,airline_iata,airline_icao,aircraft_model,aircraft_reg,aircraft_mode_s,quality_dep,quality_arr,aircraft_family,num_seats,is_widebody,is_narrowbody,is_regional,aircraft_size_category,is_freighter,is_holiday,vac_school,is_holiday_eve,is_holiday_next,is_weekend,is_weekend_or_holiday,holiday_name,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_gusts_10m,pressure_msl,precipitation,cloud_cover
0,2026-04-10,LFMN,arrival,IB 1217,None,Expected,IsOperator,False,2026-04-09 22:05:00,None,None,NaN,4,1,K,None,None,None,None,Iberia,IB,IBE,Bombardier CRJ1000,None,None,"[Basic, Live]",[Basic],Bombardier CRJ,85,False,False,True,Small,False,0,1,0,0,0,0,None,14.5,91,4.3,9.7,1015.5,0.0,42
1,2026-04-10,LFMN,arrival,TP 484,None,Expected,IsOperator,False,2026-04-09 22:05:00,None,None,NaN,1,1,None,None,None,None,None,TAP Air Portugal,TP,TAP,Embraer 190,None,None,"[Basic, Live]",[Basic],Embraer E-Jet,100,False,True,True,Small,False,0,1,0,0,0,0,None,14.5,91,4.3,9.7,1015.5,0.0,42
2,2026-04-10,LFLL,arrival,V7 2919,None,Expected,IsOperator,False,2026-04-09 22:30:00,2026-04-09 22:30Z,None,0.0,PAX,1,D,None,None,None,None,Volotea,V7,VOE,Airbus A320,None,None,"[Basic, Live]","[Basic, Live]",Airbus A320 Family,165,False,True,False,Medium,False,0,1,0,0,0,0,None,13.7,67,3.9,7.9,1018.2,0.0,0
3,2026-04-10,LFPG,departure,LY 326,None,Expected,Unknown,False,2026-04-09 22:35:00,None,None,NaN,2B,None,None,None,None,None,Tel Aviv Yafo,El Al,LY,ELY,Boeing 787-9,None,None,[Basic],[],Boeing 787,295,True,False,False,Very Large,False,0,1,0,0,0,0,None,10.0,68,13.2,31.7,1021.5,0.0,100
4,2026-04-10,LFMN,arrival,U2 1616,None,Expected,IsOperator,False,2026-04-09 22:40:00,None,None,NaN,None,2,None,None,None,None,None,easyJet,U2,EZY,Airbus A320,None,None,[Basic],[Basic],Airbus A320 Family,165,False,True,False,Medium,False,0,1,0,0,0,0,None,14.2,91,4.1,7.9,1015.1,0.0,54


In [14]:
#colonnes avec valeurs manquantes
df_predict.isna().sum()
df_predict.isna().sum() / len(df_ppml) * 100
#colonnes avec string 'None' envoyées par l'api en string:
#on va remplacer les string 'None' par le vrai Nan:
df_predict = df_predict.astype(str).replace(
    ['None', 'none', 'NULL', 'null', '', 'NaN', 'nan'], 
    np.nan
)

/var/folders/p7/7w66qcfn4yj82k3sbs5fp74h0000gn/T/ipykernel_74878/2295560977.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_predict = df_predict.astype(str).replace(


In [111]:
df_ppml.icao.nunique()                   

5

In [112]:
df_ppml.isna().sum() / len(df_ppml) * 100

date                       0.000000
icao                       0.000000
type                       0.000000
flight_number              0.000000
call_sign                 44.835492
status                     0.000000
codeshare_status           0.000000
is_cargo                   0.000000
scheduled_utc              0.000000
revised_utc                5.463082
runway_utc                88.357714
delay_minutes              5.463082
terminal_dep              20.097145
terminal_arr              19.644658
gate_dep                  78.859264
baggage_belt              84.859816
destination_icao          50.985991
destination_iata          50.985991
destination_name          50.865882
airline_name               0.000000
airline_iata               0.798840
airline_icao               0.783354
aircraft_model             0.335022
aircraft_reg              40.593899
aircraft_mode_s           39.874754
quality_dep                0.000000
quality_arr                0.000000
aircraft_family            0

### attention cas particuliers de previsions de mouvements aeroports!
on veut le status "Expected" sinon il a déjà été annulé, Retardé, ou erreur = arrivé pour une date dans le futur

In [113]:
df_predict['status'].value_counts()

status
Expected    35432
Canceled      165
Delayed         4
Arrived         3
Name: count, dtype: int64

In [114]:
df_predict[df_predict['status'] == 'Arrived']

,date,icao,type,flight_number,call_sign,status,codeshare_status,is_cargo,scheduled_utc,revised_utc,runway_utc,delay_minutes,terminal_dep,terminal_arr,gate_dep,baggage_belt,destination_icao,destination_iata,destination_name,airline_name,airline_iata,airline_icao,aircraft_model,aircraft_reg,aircraft_mode_s,quality_dep,quality_arr,aircraft_family,num_seats,is_widebody,is_narrowbody,is_regional,aircraft_size_category,is_freighter,is_holiday,vac_school,is_holiday_eve,is_holiday_next,is_weekend,is_weekend_or_holiday,holiday_name,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_gusts_10m,pressure_msl,precipitation,cloud_cover
651,2026-04-10,LFPG,arrival,KL 2273,NaN,Arrived,IsCodeshared,False,2026-04-10 06:05:00,2026-04-10 05:44Z,NaN,-21.0,NaN,2E,NaN,NaN,NaN,NaN,NaN,KLM,KL,KLM,Boeing 777-200,NaN,NaN,['Basic'],['Basic' 'Live'],Boeing 777,355,True,False,False,Very Large,False,0,1,0,0,0,0,NaN,7.4,72,9.7,19.8,1023.2,0.0,100
652,2026-04-10,LFPG,arrival,DL 8205,NaN,Arrived,IsCodeshared,False,2026-04-10 06:05:00,2026-04-10 05:44Z,NaN,-21.0,NaN,2E,NaN,NaN,NaN,NaN,NaN,Delta Air Lines,DL,DAL,Boeing 777-200,NaN,NaN,['Basic'],['Basic' 'Live'],Boeing 777,355,True,False,False,Very Large,False,0,1,0,0,0,0,NaN,7.4,72,9.7,19.8,1023.2,0.0,100
659,2026-04-10,LFPG,arrival,AF 805,NaN,Arrived,IsOperator,False,2026-04-10 06:05:00,2026-04-10 05:44Z,NaN,-21.0,NaN,2E,NaN,NaN,NaN,NaN,NaN,Air France,AF,AFR,Boeing 777-200,NaN,NaN,['Basic'],['Basic' 'Live'],Boeing 777,355,True,False,False,Very Large,False,0,1,0,0,0,0,NaN,7.4,72,9.7,19.8,1023.2,0.0,100


In [15]:
df_predict.isna().sum() / len(df_ppml) * 100

date                       0.000000
icao                       0.000000
type                       0.000000
flight_number              0.000000
call_sign                 13.352521
status                     0.000000
codeshare_status           0.000000
is_cargo                   0.000000
scheduled_utc              0.000000
revised_utc                1.889643
runway_utc                13.447701
delay_minutes              1.889643
terminal_dep               3.110754
terminal_arr               3.106977
gate_dep                  13.155360
baggage_belt              13.112302
destination_icao           6.775974
destination_iata           6.775974
destination_name           6.754067
airline_name               0.000000
airline_iata               0.102357
airline_icao               0.107645
aircraft_model             0.046080
aircraft_reg              13.400111
aircraft_mode_s           13.400111
quality_dep                0.000000
quality_arr                0.000000
aircraft_family            0

In [116]:
df_ppml["flight_number"].value_counts()

flight_number
AF 7300     97
KL 2078     96
AF 7313     96
AF 7371     95
AF 7302     95
AF 7301     93
AF 7304     92
AF 7362     88
AF 7330     82
U2 1631     82
AF 7343     82
AF 7363     80
DL 8495     80
AY 6312     79
U2 1632     79
AF 7331     79
AM 5923     79
AF 7360     78
AF 7336     77
U2 4860     77
U2 4859     77
AF 7305     77
FB 1562     77
VN 3183     77
AF 7310     76
DL 8340     76
AF 7312     76
KQ 3007     76
MU 1509     75
AF 7306     75
KE 6351     75
AF 7307     75
AM 5752     74
AF 1449     73
KQ 3869     73
G3 5027     73
KL 2255     72
EK 71       72
AF 7311     72
AF 7303     71
AF 7344     71
VN 3200     70
G3 5020     70
AF 1733     70
AF 6202     70
AF 1107     70
AF 1415     70
AI 147      70
AF 1212     69
VN 3199     69
AF 1401     69
TS 110      69
AF 7495     69
AF 7520     69
DL 228      69
DL 8193     69
AF 1385     69
AF 7432     69
DL 8330     69
AF 1497     68
AF 257      68
AF 1269     68
AF 51       68
AH 1231     68
AF 1240     68
AF 1680    

### CONCLUSION:


# 3. Nettoyage des données

In [117]:
df_ppml.columns

Index(['date', 'icao', 'type', 'flight_number', 'call_sign', 'status',
       'codeshare_status', 'is_cargo', 'scheduled_utc', 'revised_utc',
       'runway_utc', 'delay_minutes', 'terminal_dep', 'terminal_arr',
       'gate_dep', 'baggage_belt', 'destination_icao', 'destination_iata',
       'destination_name', 'airline_name', 'airline_iata', 'airline_icao',
       'aircraft_model', 'aircraft_reg', 'aircraft_mode_s', 'quality_dep',
       'quality_arr', 'aircraft_family', 'num_seats', 'is_widebody',
       'is_narrowbody', 'is_regional', 'aircraft_size_category',
       'is_freighter', 'is_holiday', 'vac_school', 'is_holiday_eve',
       'is_holiday_next', 'is_weekend', 'is_weekend_or_holiday',
       'holiday_name', 'temperature_2m', 'relative_humidity_2m',
       'wind_speed_10m', 'wind_gusts_10m', 'pressure_msl', 'precipitation',
       'cloud_cover'],
      dtype='object')

In [16]:
null_dest = df_ppml[df_ppml['destination_icao'].isna()]

print(f"Nombre de lignes avec destination_icao null : {len(null_dest)}\n")
print("Aperçu des 10 premières lignes avec destination_icao null :")
display(null_dest.head(10))

Nombre de lignes avec destination_icao null : 134990

Aperçu des 10 premières lignes avec destination_icao null :


,date,icao,type,flight_number,call_sign,status,codeshare_status,is_cargo,scheduled_utc,revised_utc,runway_utc,delay_minutes,terminal_dep,terminal_arr,gate_dep,baggage_belt,destination_icao,destination_iata,destination_name,airline_name,airline_iata,airline_icao,aircraft_model,aircraft_reg,aircraft_mode_s,quality_dep,quality_arr,aircraft_family,num_seats,is_widebody,is_narrowbody,is_regional,aircraft_size_category,is_freighter,is_holiday,vac_school,is_holiday_eve,is_holiday_next,is_weekend,is_weekend_or_holiday,holiday_name,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_gusts_10m,pressure_msl,precipitation,cloud_cover
11,2025-10-05,LFPG,departure,E4 76JE,ENT76JE,Departed,IsOperator,False,2025-10-05 03:15:00,2025-10-05 03:15Z,2025-10-05 03:15Z,0.0,None,None,None,None,None,None,Unknown,Enter Air,E4,ENT,Boeing 737-800,SP-ESI,489228,"[Basic, Live]",[],Boeing 737 Family,155,False,True,False,Medium,False,0,0,0,0,1,1,None,9.9,83,17.6,34.6,1015.2,0.0,13
12,2025-10-05,LFPG,arrival,CMA 583,CMA583,Approaching,IsOperator,False,2025-10-05 03:25:00,2025-10-05 03:25Z,None,0.0,None,None,None,None,None,None,None,CMA,None,None,Boeing 777,F-HMRB,39B221,"[Basic, Live]","[Basic, Live]",Boeing 777,355,True,False,False,Very Large,False,0,0,0,0,1,1,None,9.9,83,17.6,34.6,1015.2,0.0,13
13,2025-10-05,LFPG,arrival,UU 973,REU973,Approaching,IsOperator,False,2025-10-05 03:30:00,2025-10-05 03:31Z,None,1.0,1A,2B,None,None,None,None,None,Air Austral,UU,REU,Boeing 787-8,F-OLRC,3A2E22,[Basic],"[Basic, Live]",Boeing 787,295,True,False,False,Very Large,False,0,0,0,0,1,1,None,9.9,83,17.6,34.6,1015.2,0.0,13
14,2025-10-05,LFPG,arrival,UU 975,REU975,Approaching,IsOperator,False,2025-10-05 03:30:00,2025-10-05 03:38Z,None,8.0,None,2B,None,None,None,None,None,Air Austral,UU,REU,Boeing 777-300,F-OLRE,3A2E24,[Basic],"[Basic, Live]",Boeing 777,355,True,False,False,Very Large,False,0,0,0,0,1,1,None,9.9,83,17.6,34.6,1015.2,0.0,13
15,2025-10-05,LFPG,arrival,AF 407,AFR407,Approaching,IsOperator,False,2025-10-05 03:45:00,2025-10-05 03:28Z,None,-17.0,2,2E,None,None,None,None,None,Air France,AF,AFR,Airbus A350-900,F-HTYN,39CF0D,[Basic],"[Basic, Live]",Airbus A350,325,True,False,False,Very Large,False,0,0,0,0,1,1,None,9.9,83,18.1,35.3,1015.6,0.0,24
16,2025-10-05,LFPG,arrival,AF 771,None,Unknown,IsOperator,False,2025-10-05 03:45:00,None,None,NaN,None,2E,None,None,None,None,None,Air France,AF,AFR,Boeing 777-300ER,None,None,[],[Basic],Boeing 777,355,True,False,False,Very Large,False,0,0,0,0,1,1,None,9.9,83,18.1,35.3,1015.6,0.0,24
17,2025-10-05,LFPG,arrival,AF 841,None,Unknown,IsOperator,False,2025-10-05 03:45:00,None,None,NaN,None,2E,None,None,None,None,None,Air France,AF,AFR,Boeing 777-300ER,None,None,[],[Basic],Boeing 777,355,True,False,False,Very Large,False,0,0,0,0,1,1,None,9.9,83,18.1,35.3,1015.6,0.0,24
18,2025-10-05,LFPG,arrival,AF 343,AFR343G,Expected,IsOperator,False,2025-10-05 03:45:00,None,None,NaN,None,2E,A57,None,None,None,None,Air France,AF,AFR,Boeing 787-9,F-HRBC,39C422,"[Basic, Live]",[Basic],Boeing 787,295,True,False,False,Very Large,False,0,0,0,0,1,1,None,9.9,83,18.1,35.3,1015.6,0.0,24
19,2025-10-05,LFPG,arrival,AF 223,AFR223,Unknown,IsOperator,False,2025-10-05 03:45:00,None,None,NaN,3,2E,None,None,None,None,None,Air France,AF,AFR,Airbus A350,F-HTYC,39CF02,[Basic],[Basic],Airbus A350,325,True,False,False,Very Large,False,0,0,0,0,1,1,None,9.9,83,18.1,35.3,1015.6,0.0,24
21,2025-10-05,LFPG,arrival,AF 185,AFR185,Expected,IsOperator,False,2025-10-05 03:50:00,None,None,NaN,1,2E,8,None,None,None,None,Air France,AF,AFR,Airbus A350-900,F-HUVL,39D2AB,"[Basic, Live]",[Basic],Airbus A350,325,True,False,False,Very Large,False,0,0,0,0,1,1,None,9.9,83,18.1,35.3,1015.6,0.0,24


In [17]:
# =============================================
# 3. NETTOYAGE DES DONNÉES
# =============================================

print(f"Shape initial du dataset : {df_ppml.shape}")

# 1. Suppression des vols sans information de retard (important)
df_clean = df_ppml.dropna(subset=['delay_minutes']).copy()

# 2. l'api ne remplit l'aeroport de destination (destination_icao) que pour les départs car les arrivées sont deduites de la colonne icao
# on remet l'information pour le modele
df_clean['dest_icao_clean'] = df_clean.apply(
    lambda row: row['icao'] if row['type'] == "arrival" else row['destination_icao'], 
    axis=1
)

print(f"Après suppression des lignes sans delay_minutes : {df_clean.shape[0]:,} lignes "
      f"({(df_clean.shape[0]/df_ppml.shape[0]*100):.1f}% conservés)")

# 3. Traitement des valeurs manquantes restantes

# a) Variables météo => imputation par la médiane
meteo_cols = ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m',
              'wind_gusts_10m', 'pressure_msl', 'precipitation', 'cloud_cover']

for col in meteo_cols:
    if col in df_clean.columns:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)


# dest_icao_clean (très peu de NaN)
df_clean['dest_icao_clean'] = df_clean['dest_icao_clean'].fillna('UNKNOWN')

# terminal_dep et terminal_arr (~20% NaN → on crée une vraie catégorie "inconnue")
df_clean['terminal_dep'] = df_clean['terminal_dep'].fillna('UNKNOWN')
df_clean['terminal_arr'] = df_clean['terminal_arr'].fillna('UNKNOWN')

# aircraft_model (seulement 0.35% NaN → on prend la valeur la plus courante)
most_common_model = df_clean['aircraft_model'].mode()[0]
df_clean['aircraft_model'] = df_clean['aircraft_model'].fillna(most_common_model)

print("NaN restants :")
print(df_clean.isna().mean().sort_values(ascending=False))
df_clean.head()

Shape initial du dataset : (264759, 48)
Après suppression des lignes sans delay_minutes : 250,295 lignes (94.5% conservés)
NaN restants :
holiday_name              0.957774
runway_utc                0.877105
baggage_belt              0.845406
gate_dep                  0.785150
destination_iata          0.506658
destination_icao          0.506658
destination_name          0.506175
call_sign                 0.442678
aircraft_reg              0.399325
aircraft_mode_s           0.391914
airline_iata              0.008338
airline_icao              0.008158
temperature_2m            0.000000
aircraft_size_category    0.000000
is_freighter              0.000000
is_holiday                0.000000
vac_school                0.000000
cloud_cover               0.000000
is_holiday_eve            0.000000
is_holiday_next           0.000000
relative_humidity_2m      0.000000
precipitation             0.000000
pressure_msl              0.000000
wind_gusts_10m            0.000000
wind_speed_10m        

,date,icao,type,flight_number,call_sign,status,codeshare_status,is_cargo,scheduled_utc,revised_utc,runway_utc,delay_minutes,terminal_dep,terminal_arr,gate_dep,baggage_belt,destination_icao,destination_iata,destination_name,airline_name,airline_iata,airline_icao,aircraft_model,aircraft_reg,aircraft_mode_s,quality_dep,quality_arr,aircraft_family,num_seats,is_widebody,is_narrowbody,is_regional,aircraft_size_category,is_freighter,is_holiday,vac_school,is_holiday_eve,is_holiday_next,is_weekend,is_weekend_or_holiday,holiday_name,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_gusts_10m,pressure_msl,precipitation,cloud_cover,dest_icao_clean
0,2025-10-05,LFPG,departure,BJ 521,LBT521,Departed,IsOperator,False,2025-10-04 20:05:00,2025-10-04 23:28Z,2025-10-04 23:28Z,203.0,3,UNKNOWN,None,None,DTTA,TUN,Tunis,Nouvelair Tunisie,BJ,LBT,Airbus A320,TS-INM,02A1AC,"[Basic, Live]",[],Airbus A320 Family,165,False,True,False,Medium,False,0,0,0,0,1,1,None,10.6,75,19.6,41.4,1013.8,0.0,23,DTTA
1,2025-10-05,LFPG,departure,BJ 543,LBT543,Departed,IsOperator,False,2025-10-04 20:15:00,2025-10-04 22:35Z,2025-10-04 22:35Z,140.0,3,UNKNOWN,None,02,DTMB,MIR,Monastir,Nouvelair Tunisie,BJ,LBT,Airbus A320,TS-INR,02A1B1,"[Basic, Live]","[Basic, Live]",Airbus A320 Family,165,False,True,False,Medium,False,0,0,0,0,1,1,None,10.6,75,19.6,41.4,1013.8,0.0,23,DTMB
2,2025-10-05,LFPG,departure,AF 406,AFR406,Departed,IsOperator,False,2025-10-04 21:20:00,2025-10-04 22:14Z,2025-10-04 22:14Z,54.0,2E,2,None,None,SCEL,SCL,Santiago,Air France,AF,AFR,Airbus A350-900,F-HTYT,39CF13,"[Basic, Live]",[Basic],Airbus A350,325,True,False,False,Very Large,False,0,0,0,0,1,1,None,10.4,77,19.9,38.9,1014.2,0.0,15,SCEL
3,2025-10-05,LFPG,departure,AF 454,AFR454,Departed,IsOperator,False,2025-10-04 21:30:00,2025-10-04 22:03Z,2025-10-04 22:03Z,33.0,2E,3,None,None,SBGR,GRU,São Paulo,Air France,AF,AFR,Boeing 777-300,F-GSQY,394A18,"[Basic, Live]","[Basic, Live]",Boeing 777,355,True,False,False,Very Large,False,0,0,0,0,1,1,None,10.4,77,19.9,38.9,1014.2,0.0,15,SBGR
4,2025-10-05,LFPG,departure,AF 116,AFR116,Departed,IsOperator,False,2025-10-04 21:45:00,2025-10-04 22:04Z,2025-10-04 22:04Z,19.0,2E,1,None,13,ZSPD,PVG,Shanghai,Air France,AF,AFR,Boeing 777,F-GZNS,3965B2,"[Basic, Live]","[Basic, Live]",Boeing 777,355,True,False,False,Very Large,False,0,0,0,0,1,1,None,10.1,77,19.9,38.9,1014.4,0.0,20,ZSPD


In [18]:
df_clean.isna().sum() / len(df_clean) * 100

date                       0.000000
icao                       0.000000
type                       0.000000
flight_number              0.000000
call_sign                 44.267764
status                     0.000000
codeshare_status           0.000000
is_cargo                   0.000000
scheduled_utc              0.000000
revised_utc                0.000000
runway_utc                87.710502
delay_minutes              0.000000
terminal_dep               0.000000
terminal_arr               0.000000
gate_dep                  78.514952
baggage_belt              84.540642
destination_icao          50.665814
destination_iata          50.665814
destination_name          50.617471
airline_name               0.000000
airline_iata               0.833816
airline_icao               0.815837
aircraft_model             0.000000
aircraft_reg              39.932480
aircraft_mode_s           39.191354
quality_dep                0.000000
quality_arr                0.000000
aircraft_family            0

In [19]:
# # =============================================
# # 4. GESTION DES OUTLIERS sur delay_minutes
# # =============================================

# Version propre et recommandée
lower = -30
upper = 150

df_clean['delay_minutes_clean'] = df_clean['delay_minutes'].clip(lower=lower, upper=upper)

print("Statistiques après clipping recommandé :")
print(df_clean['delay_minutes_clean'].describe())

# Pourcentage de valeurs qui ont été modifiées
modified = (df_clean['delay_minutes'] != df_clean['delay_minutes_clean']).mean() * 100
print(f"\nPourcentage de lignes modifiées : {modified:.3f} %")

Statistiques après clipping recommandé :
count    250295.000000
mean         15.464927
std          28.702377
min         -30.000000
25%          -3.000000
50%          12.000000
75%          26.000000
max         150.000000
Name: delay_minutes_clean, dtype: float64

Pourcentage de lignes modifiées : 2.005 %


In [20]:
print("NaN restants :")
print(df_clean.isna().mean().sort_values(ascending=False))

NaN restants :
holiday_name              0.957774
runway_utc                0.877105
baggage_belt              0.845406
gate_dep                  0.785150
destination_icao          0.506658
destination_iata          0.506658
destination_name          0.506175
call_sign                 0.442678
aircraft_reg              0.399325
aircraft_mode_s           0.391914
airline_iata              0.008338
airline_icao              0.008158
is_holiday_eve            0.000000
vac_school                0.000000
is_holiday                0.000000
is_weekend                0.000000
is_freighter              0.000000
is_holiday_next           0.000000
date                      0.000000
is_regional               0.000000
is_weekend_or_holiday     0.000000
temperature_2m            0.000000
relative_humidity_2m      0.000000
wind_speed_10m            0.000000
wind_gusts_10m            0.000000
pressure_msl              0.000000
precipitation             0.000000
cloud_cover               0.000000
dest_

Résumé des stats :
l y a des outliers très extrêmes des deux côtés :

Des avances absurdes (jusqu’à -695 min)  
Des retards extrêmes (jusqu’à +1233 min)

Nombre de lignes : 250 295  
Moyenne : +15.81 min (retard moyen)  
Médiane : +12 min (la moitié des vols ont plus de 12 min de retard)  
Écart-type : 32.45 min => très grande dispersion  
Min : -695 min (avance énorme, presque 12 heures !)  
Max : +1233 min (~ +20.5 heures de retard)  
75% des vols ≤ +26 min  

### Bilan du nettoyage


### Traitement des outliers :


# 4. Feature Engineering

### 1.  Création de flight_key, airline_group  
"AF 7313" => "AF" + "7313" + "AF_7313"  
Regroupement des compagnies rares Comme dans le cours sur TITANIC (but 70 à 90 companies au total)

In [21]:
import numpy as np
import pandas as pd

df_fe = df_clean.copy()

# 1. Convertir en datetime
df_fe['scheduled_utc'] = pd.to_datetime(df_fe['scheduled_utc'])

# 2. Features temporelles de base
df_fe['dep_hour']       = df_fe['scheduled_utc'].dt.hour
df_fe['dep_dayofweek']  = df_fe['scheduled_utc'].dt.dayofweek
df_fe['dep_dayofmonth'] = df_fe['scheduled_utc'].dt.day
df_fe['dep_month']      = df_fe['scheduled_utc'].dt.month

# 3. Encodage cyclique 
df_fe['dep_hour_sin'] = np.sin(2 * np.pi * df_fe['dep_hour'] / 24)
df_fe['dep_hour_cos'] = np.cos(2 * np.pi * df_fe['dep_hour'] / 24)

df_fe['dep_dayofweek_sin'] = np.sin(2 * np.pi * df_fe['dep_dayofweek'] / 7)
df_fe['dep_dayofweek_cos'] = np.cos(2 * np.pi * df_fe['dep_dayofweek'] / 7)

# 4. Période de la journée 
df_fe['period_of_day'] = pd.cut(
    df_fe['dep_hour'], 
    bins=[0, 6, 10, 16, 20, 24], 
    labels=['night', 'morning', 'afternoon', 'evening', 'late_night'],   # 'night' → 'late_night'
    right=False,
    ordered=False   # Important labels identiques 
)


print("Feature Engineering terminé.")
print(f"Shape final : {df_fe.shape}")
print(f"Colonnes ajoutées : features temporelles")

Feature Engineering terminé.
Shape final : (250295, 59)
Colonnes ajoutées : features temporelles


In [22]:
# Suppression des colonnes inutiles ou trop incomplètes (optionnel, à ajuster selon les besoins)
colonnes_a_dropper = [
    "scheduled_utc",
    "flight_number",
    "date",
    "destination_icao",
    "destination_iata",
    "destination_name",
    "holiday_name",
    "runway_utc",
    "call_sign",
    "aircraft_reg",
    "aircraft_mode_s",
    "airline_iata",
    "airline_icao",
    "is_weekend_or_holiday",
    #leakage potentiel
    'status', 
    'delay_minutes',# cible remplacée par delay_minutes_clean 
    'revised_utc', 
    'runway_utc', 
    'gate_dep', 
    'baggage_belt', 
    'quality_dep', 
    'quality_arr'
]

df_fe = df_fe.drop(columns=colonnes_a_dropper)

In [23]:
print("Colonnes categorielles :")
print(df_fe.select_dtypes(include=['object', 'category']).nunique())
print("Colonnes numériques :")
print(df_fe.select_dtypes(include=np.number).columns.tolist())

Colonnes categorielles :
icao                        5
type                        2
codeshare_status            3
terminal_dep               38
terminal_arr               41
airline_name              256
aircraft_model            124
aircraft_family            16
aircraft_size_category      4
dest_icao_clean           365
period_of_day               5
dtype: int64
Colonnes numériques :
['num_seats', 'is_holiday', 'vac_school', 'is_holiday_eve', 'is_holiday_next', 'is_weekend', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'wind_gusts_10m', 'pressure_msl', 'precipitation', 'cloud_cover', 'delay_minutes_clean', 'dep_hour', 'dep_dayofweek', 'dep_dayofmonth', 'dep_month', 'dep_hour_sin', 'dep_hour_cos', 'dep_dayofweek_sin', 'dep_dayofweek_cos']


In [25]:
df_fe.to_parquet(path="/Users/manjakaranjatoson/Desktop/A_I/JEDHA/PROJECTS/CDSD/PPML/data/processed/test_patrick/df_fe.parquet")
print("Export successful")

Export successful


## 6. Choix du Modèle : Recherche d'une Baseline



### Comparaison des Modèles de Base



# 6. Modélisation

In [126]:
# from catboost import CatBoostRegressor, Pool
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# import numpy as np

# df_model = df_fe.copy()

# # Suppression propre de la cible
# X = df_model.drop(columns=['delay_minutes_clean'], errors='ignore')
# y = df_model['delay_minutes_clean']

# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=42)

# categorical_features = [
#     'icao', 'type',  'codeshare_status',
#     'terminal_dep', 'terminal_arr', 'airline_name',
#     'aircraft_model', 'aircraft_family', 'aircraft_size_category',
#     'dest_icao_clean', 'period_of_day'
# ]

# train_pool = Pool(X_train, y_train, cat_features=categorical_features)
# val_pool   = Pool(X_val,   y_val,   cat_features=categorical_features)

# model = CatBoostRegressor(
#     iterations=6000,
#     learning_rate=0.06,           # un peu plus bas pour mieux converger
#     depth=8,
#     loss_function='RMSE',
#     eval_metric='RMSE',            # on garde RMSE comme métrique d'évaluation
#     random_seed=42,
#     early_stopping_rounds=300,    # augmenté
#     verbose=200,
#     task_type='GPU',              
#     l2_leaf_reg=3,                # régularisation
#     random_strength=1
# )
# print("\nEntraînement version optimisée...\n")
# model.fit(train_pool, eval_set=val_pool, use_best_model=True)

# # Évaluation
# preds = model.predict(X_val)
# mae = mean_absolute_error(y_val, preds)
# rmse = np.sqrt(mean_squared_error(y_val, preds))
# r2 = r2_score(y_val, preds)

# print(f"MAE   : {mae:.3f} minutes")
# print(f"RMSE  : {rmse:.3f} minutes")
# print(f"R²    : {r2:.4f}")
# print(f"Best iteration : {model.get_best_iteration()}")

In [127]:
import pandas as pd

# Feature Importance
fi = model.get_feature_importance()
feature_names = X_train.columns

fi_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': fi
}).sort_values(by='Importance', ascending=False)

print("Feature Importance :\n")
print(fi_df.round(2))


NameError: name 'model' is not defined

In [ ]:
df_fe.head()


,icao,type,codeshare_status,is_cargo,terminal_dep,terminal_arr,airline_name,aircraft_model,aircraft_family,num_seats,is_widebody,is_narrowbody,is_regional,aircraft_size_category,is_freighter,is_holiday,vac_school,is_holiday_eve,is_holiday_next,is_weekend,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_gusts_10m,pressure_msl,precipitation,cloud_cover,dest_icao_clean,delay_minutes_clean,dep_hour,dep_dayofweek,dep_dayofmonth,dep_month,dep_hour_sin,dep_hour_cos,dep_dayofweek_sin,dep_dayofweek_cos,period_of_day
0,LFPG,departure,IsOperator,False,3,UNKNOWN,Nouvelair Tunisie,Airbus A320,Airbus A320 Family,165,False,True,False,Medium,False,0,0,0,0,1,10.6,75,19.6,41.4,1013.8,0.0,23,DTTA,150.0,20,5,4,10,-0.866025,0.500000,-0.974928,-0.222521,late_night
1,LFPG,departure,IsOperator,False,3,UNKNOWN,Nouvelair Tunisie,Airbus A320,Airbus A320 Family,165,False,True,False,Medium,False,0,0,0,0,1,10.6,75,19.6,41.4,1013.8,0.0,23,DTMB,140.0,20,5,4,10,-0.866025,0.500000,-0.974928,-0.222521,late_night
2,LFPG,departure,IsOperator,False,2E,2,Air France,Airbus A350-900,Airbus A350,325,True,False,False,Very Large,False,0,0,0,0,1,10.4,77,19.9,38.9,1014.2,0.0,15,SCEL,54.0,21,5,4,10,-0.707107,0.707107,-0.974928,-0.222521,late_night
3,LFPG,departure,IsOperator,False,2E,3,Air France,Boeing 777-300,Boeing 777,355,True,False,False,Very Large,False,0,0,0,0,1,10.4,77,19.9,38.9,1014.2,0.0,15,SBGR,33.0,21,5,4,10,-0.707107,0.707107,-0.974928,-0.222521,late_night
4,LFPG,departure,IsOperator,False,2E,1,Air France,Boeing 777,Boeing 777,355,True,False,False,Very Large,False,0,0,0,0,1,10.1,77,19.9,38.9,1014.4,0.0,20,ZSPD,19.0,21,5,4,10,-0.707107,0.707107,-0.974928,-0.222521,late_night


In [ ]:
# from category_encoders import TargetEncoder
# from catboost import CatBoostRegressor, Pool
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# import numpy as np

# # =============================================
# # 1. Préparation
# # =============================================
# df_model = df_fe.copy()

# X = df_model.drop(columns=['delay_minutes_clean'])
# y = df_model['delay_minutes_clean']

# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=42)

# # =============================================
# # 2. Target Encoding (Correct)
# # =============================================
# high_card_cols = ['airline_name', 'dest_icao_clean', 'aircraft_model']

# te = TargetEncoder(cols=high_card_cols, smoothing=10, min_samples_leaf=5)

# # Fit + Transform sur le train
# X_train = te.fit_transform(X_train, y_train)

# # Transform sur la validation
# X_val = te.transform(X_val)

# print("Target Encoding terminé sur airline_name, dest_icao_clean et aircraft_model")

# # =============================================
# # 3. Features catégorielles pour CatBoost (après encodage)
# # =============================================
# categorical_features = [
#     'icao', 'type', 'codeshare_status',
#     'terminal_dep', 'terminal_arr',
#     'aircraft_family', 'aircraft_size_category',
#     'period_of_day'
# ]

# # Note : airline_name, dest_icao_clean et aircraft_model sont maintenant numériques → plus dans categorical_features

# # =============================================
# # 4. Pools et Modèle
# # =============================================
# train_pool = Pool(X_train, y_train, cat_features=categorical_features)
# val_pool   = Pool(X_val,   y_val,   cat_features=categorical_features)

# model = CatBoostRegressor(
#     iterations=6000,
#     learning_rate=0.06,
#     depth=8,
#     loss_function='RMSE',
#     eval_metric='RMSE',
#     random_seed=42,
#     early_stopping_rounds=300,
#     verbose=200,
#     task_type='GPU',
#     l2_leaf_reg=3,
#     random_strength=1
# )

# print("\nEntraînement avec Target Encoding...\n")
# model.fit(train_pool, eval_set=val_pool, use_best_model=True)

# # =============================================
# # 5. Évaluation
# # =============================================
# preds = model.predict(X_val)
# mae = mean_absolute_error(y_val, preds)
# rmse = np.sqrt(mean_squared_error(y_val, preds))
# r2 = r2_score(y_val, preds)

# print(f"MAE   : {mae:.3f} minutes")
# print(f"RMSE  : {rmse:.3f} minutes")
# print(f"R²    : {r2:.4f}")
# print(f"Best iteration : {model.get_best_iteration()}")

Target Encoding terminé sur airline_name, dest_icao_clean et aircraft_model

Entraînement avec Target Encoding...

0:	learn: 28.2584650	test: 28.2680393	best: 28.2680393 (0)	total: 32.4ms	remaining: 3m 14s
200:	learn: 22.5215009	test: 22.7670437	best: 22.7670437 (200)	total: 6.57s	remaining: 3m 9s
400:	learn: 21.7975536	test: 22.2221899	best: 22.2221899 (400)	total: 13.2s	remaining: 3m 3s
600:	learn: 21.2394624	test: 21.8289900	best: 21.8289900 (600)	total: 19.5s	remaining: 2m 55s
800:	learn: 20.7501039	test: 21.4901941	best: 21.4901941 (800)	total: 25.8s	remaining: 2m 47s
1000:	learn: 20.2983963	test: 21.1915927	best: 21.1915927 (1000)	total: 32s	remaining: 2m 39s
1200:	learn: 19.8785451	test: 20.9171565	best: 20.9171565 (1200)	total: 38.4s	remaining: 2m 33s
1400:	learn: 19.5198584	test: 20.6822303	best: 20.6822303 (1400)	total: 45.2s	remaining: 2m 28s
1600:	learn: 19.1677996	test: 20.4621010	best: 20.4621010 (1600)	total: 52s	remaining: 2m 22s
1800:	learn: 18.8654327	test: 20.2748440

***

In [ ]:
# import pandas as pd

# # Feature Importance
# fi = model.get_feature_importance()
# feature_names = X_train.columns

# fi_df = pd.DataFrame({
#     'Feature': feature_names,
#     'Importance': fi
# }).sort_values(by='Importance', ascending=False)

# print("Feature Importance :\n")
# print(fi_df.round(2))


Feature Importance :

                   Feature  Importance
27         dest_icao_clean        8.09
1                     type        7.44
4             terminal_dep        7.36
20          temperature_2m        7.17
7           aircraft_model        5.81
5             terminal_arr        5.51
23          wind_gusts_10m        5.24
24            pressure_msl        5.21
22          wind_speed_10m        5.18
30          dep_dayofmonth        5.00
21    relative_humidity_2m        4.67
33            dep_hour_cos        3.59
31               dep_month        3.38
9                num_seats        2.84
6             airline_name        2.83
8          aircraft_family        2.66
32            dep_hour_sin        2.58
34       dep_dayofweek_sin        2.43
26             cloud_cover        2.36
28                dep_hour        2.08
36           period_of_day        1.37
13  aircraft_size_category        1.30
29           dep_dayofweek        1.17
0                     icao        1.13
35 

In [ ]:
# =============================================
# A. NETTOYAGE FINAL DES FEATURES INUTILES
# =============================================

df_model = df_fe.copy()

# Features à supprimer (importance très faible ou nulle)
cols_to_drop = [
    "is_freighter", "is_cargo", "is_narrowbody", "is_regional",
    "is_holiday_eve", "is_holiday_next", "is_holiday", "is_weekend", 
    "vac_school", "is_holiday_eve", "is_holiday_next"   # doublons
]

df_model = df_model.drop(columns=cols_to_drop, errors='ignore')

print(f"Shape après suppression des features inutiles : {df_model.shape}")
print("Colonnes restantes :", df_model.columns.tolist())

Shape après suppression des features inutiles : (250295, 29)
Colonnes restantes : ['icao', 'type', 'codeshare_status', 'terminal_dep', 'terminal_arr', 'airline_name', 'aircraft_model', 'aircraft_family', 'num_seats', 'is_widebody', 'aircraft_size_category', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'wind_gusts_10m', 'pressure_msl', 'precipitation', 'cloud_cover', 'dest_icao_clean', 'delay_minutes_clean', 'dep_hour', 'dep_dayofweek', 'dep_dayofmonth', 'dep_month', 'dep_hour_sin', 'dep_hour_cos', 'dep_dayofweek_sin', 'dep_dayofweek_cos', 'period_of_day']


In [ ]:
# from category_encoders import TargetEncoder
# from catboost import CatBoostRegressor, Pool
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# import numpy as np

# # =============================================
# # 1. Préparation finale
# # =============================================
# df_model = df_fe.copy()

# # Suppression des features très peu utiles
# cols_inutiles = [
#     "is_freighter", "is_cargo", "is_narrowbody", "is_regional",
#     "is_holiday_eve", "is_holiday_next", "is_holiday", "is_weekend", "vac_school" # , "is_holiday_eve", "is_holiday_next"
# ]

# df_model = df_model.drop(columns=cols_inutiles, errors='ignore')

# X = df_model.drop(columns=['delay_minutes_clean'])
# y = df_model['delay_minutes_clean'] # cible

# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=42)

# # =============================================
# # 2. Target Encoding
# # =============================================
# high_card_cols = ['airline_name', 'dest_icao_clean']   # On teste sans target encoding pour aircraft_model

# te = TargetEncoder(cols=high_card_cols, smoothing=10, min_samples_leaf=5)

# # Fit + Transform
# X_train = te.fit_transform(X_train, y_train)
# X_val   = te.transform(X_val)

# # Suppression des anciennes colonnes (maintenant encodées en numérique)
# X_train = X_train.drop(columns=high_card_cols, errors='ignore')
# X_val   = X_val.drop(columns=high_card_cols, errors='ignore')

# print("Target Encoding terminé sur airline_name et dest_icao_clean")

# # =============================================
# # 3. Détection automatique des features catégorielles
# # =============================================
# categorical_features = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

# # Sécurité : on enlève les colonnes qui ont été encodées
# categorical_features = [col for col in categorical_features if col not in high_card_cols]

# print("Categorical features finales envoyées à CatBoost :", categorical_features)
# print(f"Nombre de features catégorielles : {len(categorical_features)}")

# # =============================================
# # 4. Pools CatBoost
# # =============================================
# train_pool = Pool(X_train, y_train, cat_features=categorical_features)
# val_pool   = Pool(X_val,   y_val,   cat_features=categorical_features)

# # =============================================
# # 5. Modèle final optimisé
# # =============================================
# model = CatBoostRegressor(
#     iterations=10000,
#     learning_rate=0.045,
#     depth=8,
#     loss_function='RMSE',
#     eval_metric='RMSE',
#     random_seed=42,
#     early_stopping_rounds=500,
#     verbose=200,
#     task_type='GPU',
#     l2_leaf_reg=5,
#     random_strength=1.0,
#     bagging_temperature=0.7
# )

# print("\n=== Entraînement Version Finale Optimisée ===\n")

# model.fit(train_pool, eval_set=val_pool, use_best_model=True)

# # =============================================
# # 6. Évaluation
# # =============================================
# preds = model.predict(X_val)
# mae = mean_absolute_error(y_val, preds)
# rmse = np.sqrt(mean_squared_error(y_val, preds))
# r2 = r2_score(y_val, preds)

# print(f"\nMAE   : {mae:.3f} minutes")
# print(f"RMSE  : {rmse:.3f} minutes")
# print(f"R²    : {r2:.4f}")
# print(f"Best iteration : {model.get_best_iteration()}")

Target Encoding terminé sur airline_name et dest_icao_clean
Categorical features finales envoyées à CatBoost : ['icao', 'type', 'codeshare_status', 'terminal_dep', 'terminal_arr', 'aircraft_model', 'aircraft_family', 'aircraft_size_category', 'period_of_day']
Nombre de features catégorielles : 9

=== Entraînement Version Finale Optimisée ===

0:	learn: 28.3667032	test: 28.3737896	best: 28.3737896 (0)	total: 34.4ms	remaining: 5m 44s
200:	learn: 22.8256375	test: 22.9700432	best: 22.9700432 (200)	total: 6.86s	remaining: 5m 34s
400:	learn: 22.2482646	test: 22.5132015	best: 22.5132015 (400)	total: 13.5s	remaining: 5m 23s
600:	learn: 21.8456555	test: 22.2172614	best: 22.2172614 (600)	total: 20.2s	remaining: 5m 15s
800:	learn: 21.4893798	test: 21.9718279	best: 21.9718279 (800)	total: 26.8s	remaining: 5m 8s
1000:	learn: 21.1618064	test: 21.7361969	best: 21.7361969 (1000)	total: 33.4s	remaining: 5m
1200:	learn: 20.8623876	test: 21.5291473	best: 21.5291473 (1200)	total: 40.1s	remaining: 4m 53s
1

# Premier envoi de modèle 

#### tag commun à tous les runs

In [ ]:
# ====================== SÉCURITÉ MLFLOW POUR NOTEBOOK ======================
# partie spécifique dans les notebooks
if mlflow.active_run():
    mlflow.end_run()

# Force la fermeture de tous les runs actifs
while mlflow.active_run():
    mlflow.end_run()

print(" Tous les runs MLflow précédents ont été fermés.\n")

# ====================== CONFIGURATION MLFLOW ======================
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000"))

EXPERIMENT_NAME = "ppml-retards-avion"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"MLflow experiment : {EXPERIMENT_NAME}\n")

# Tags globaux
mlflow.set_tag("project", "ppml_retards_avion")
mlflow.set_tag("dataset_version", "v2")
mlflow.set_tag("students", "Ludovic et Patrick")
mlflow.set_tag("phase", "modelisation")
mlflow.set_tag("model_type", "CatBoost")

🏃 View run clumsy-sheep-719 at: https://patjedhahf-mlflow-ppml.hf.space/#/experiments/789482705024277871/runs/310654a9a41741469de3e09775045a81
🧪 View experiment at: https://patjedhahf-mlflow-ppml.hf.space/#/experiments/789482705024277871
 Tous les runs MLflow précédents ont été fermés.

MLflow experiment : ppml-retards-avion



In [161]:
from category_encoders import TargetEncoder
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import mlflow
import mlflow.catboost
import os

# ====================== SÉCURITÉ MLFLOW ======================
if mlflow.active_run():
    mlflow.end_run()
while mlflow.active_run():
    mlflow.end_run()

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000"))
mlflow.set_experiment("ppml-retards-avion")

print("MLflow configuré avec succès\n")

# ====================== FONCTION AVEC MLFLOW ======================

def train_catboost(
    df,
    run_name: str = "CatBoost_Baseline",
    iterations: int = 6000,           # Nombre d'arbres (plus = modèle plus puissant)
    learning_rate: float = 0.06,      # Taux d'apprentissage : contrôle la vitesse de convergence
    depth: int = 8,                   # Profondeur des arbres (plus élevé = capture plus d'interactions)
    eval_metric: str = 'RMSE',        # Métrique utilisée pour l'early stopping et l'affichage
    early_stopping_rounds: int = 300, # Arrête l'entraînement si pas d'amélioration pendant X itérations
    l2_leaf_reg: float = 3,           # Régularisation L2 (anti-overfitting)
    random_strength: float = 1.0,     # Ajoute du bruit pour plus de robustesse
    bagging_temperature: float = 0.7  # Température de bagging pour plus de diversité dans les arbres
):
    """
    Fonction simple pour entraîner un modèle CatBoost avec logging MLflow.
    """
    mlflow.end_run()  # Assure que tous les runs précédents sont fermés
    with mlflow.start_run(run_name=run_name) as run:
        
        print(f"\n=== Début du run : {run_name} ===\n")

        # =============================================
        # 1. Préparation des données
        # =============================================
        df_model = df.copy()

        X = df_model.drop(columns=['delay_minutes_clean'])
        y = df_model['delay_minutes_clean']

        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.20, random_state=42
        )

        # =============================================
        # 2. Target Encoding (Pipeline de preprocessing)
        # =============================================
        high_card_cols = ['airline_name', 'dest_icao_clean', 'aircraft_model']
        te = TargetEncoder(cols=high_card_cols, smoothing=10, min_samples_leaf=5)
        
        X_train = te.fit_transform(X_train, y_train)
        X_val   = te.transform(X_val)

        # Suppression des anciennes colonnes
        X_train = X_train.drop(columns=high_card_cols, errors='ignore')
        X_val   = X_val.drop(columns=high_card_cols, errors='ignore')

        print("Target Encoding terminé sur airline_name, dest_icao_clean et aircraft_model")

        # Log du pipeline TargetEncoder pour pouvoir le recharger plus tard sur l'API
        mlflow.sklearn.log_model(te, artifact_path="target_encoder")

        # =============================================
        # 3. Features catégorielles
        # =============================================
        categorical_features = [
            'icao', 'type', 'codeshare_status',
            'terminal_dep', 'terminal_arr',
            'aircraft_family', 'aircraft_size_category',
            'period_of_day'
        ]

        # =============================================
        # 4. Pools et Modèle
        # =============================================
        train_pool = Pool(X_train, y_train, cat_features=categorical_features)
        val_pool   = Pool(X_val,   y_val,   cat_features=categorical_features)

        model = CatBoostRegressor(
            iterations=iterations,
            learning_rate=learning_rate,
            depth=depth,
            loss_function='RMSE',
            eval_metric=eval_metric,
            random_seed=42,
            early_stopping_rounds=early_stopping_rounds,
            verbose=200,
            task_type='GPU',
            l2_leaf_reg=l2_leaf_reg,
            random_strength=random_strength,
            bagging_temperature=bagging_temperature
        )

        print("\nEntraînement du modèle en cours...\n")
        model.fit(train_pool, eval_set=val_pool, use_best_model=True)

        # =============================================
        # 5. Évaluation
        # =============================================
        preds = model.predict(X_val)
        mae = mean_absolute_error(y_val, preds)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        r2 = r2_score(y_val, preds)

        # Logging des métriques
        mlflow.log_metric("MAE", round(mae, 4))
        mlflow.log_metric("RMSE", round(rmse, 4))
        mlflow.log_metric("R2", round(r2, 4))
        mlflow.log_metric("best_iteration", model.get_best_iteration())

        # Logging du modèle CatBoost (important pour l'API)
        # mlflow.catboost.log_model(model, artifact_path="catboost_model")
        # === LOGGING DU MODÈLE DANS LE REGISTRY ===
        model_info = mlflow.catboost.log_model(
            cb_model=model,
            artifact_path="catboost_model",
            registered_model_name="CatBoost_Delay_Prediction"   # ← Ce nom apparaîtra dans l'onglet Models
        )

        print(f"✅ Modèle enregistré dans le Model Registry : CatBoost_Delay_Prediction")
        print(f"   Version : {model_info.registered_model_version if hasattr(model_info, 'registered_model_version') else 'N/A'}")

        # Logging des paramètres du modèle
        mlflow.log_params({
            "iterations": iterations,
            "learning_rate": learning_rate,
            "depth": depth,
            "eval_metric": eval_metric,
            "early_stopping_rounds": early_stopping_rounds,
            "l2_leaf_reg": l2_leaf_reg,
            "random_strength": random_strength,
            "bagging_temperature": bagging_temperature,
            "use_target_encoding": True
        })

        print(f"\n=== Résultats : {run_name} ===")
        print(f"MAE   : {mae:.3f} minutes")
        print(f"RMSE  : {rmse:.3f} minutes")
        print(f"R²    : {r2:.4f}")
        print(f"Best iteration : {model.get_best_iteration()}")
        print("="*50 + "\n")

    return model

MLflow configuré avec succès



In [154]:
# ====================== EXEMPLES D'UTILISATION ======================
df_model = df_fe.copy()
# Exemple 1 : Baseline avec paramètres par défaut
model1 = train_catboost(df_model)



=== Début du run : CatBoost_Baseline ===

Target Encoding terminé sur airline_name, dest_icao_clean et aircraft_model

Entraînement du modèle en cours...

0:	learn: 28.2631458	test: 28.2739380	best: 28.2739380 (0)	total: 32.6ms	remaining: 3m 15s
200:	learn: 22.6117385	test: 22.8191629	best: 22.8191629 (200)	total: 6.45s	remaining: 3m 6s
400:	learn: 21.9724216	test: 22.3370250	best: 22.3370250 (400)	total: 13.9s	remaining: 3m 14s
600:	learn: 21.4532476	test: 21.9670515	best: 21.9670515 (600)	total: 20.6s	remaining: 3m 5s
800:	learn: 21.0041647	test: 21.6610734	best: 21.6610734 (800)	total: 27.1s	remaining: 2m 55s
1000:	learn: 20.5996774	test: 21.3956703	best: 21.3956703 (1000)	total: 33.5s	remaining: 2m 47s
1200:	learn: 20.2612761	test: 21.1870750	best: 21.1870750 (1200)	total: 39.8s	remaining: 2m 39s
1400:	learn: 19.9675449	test: 21.0051823	best: 21.0051034 (1399)	total: 46.7s	remaining: 2m 33s
1600:	learn: 19.6840553	test: 20.8382285	best: 20.8382199 (1599)	total: 53.4s	remaining: 2m

2026/04/10 00:15:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



=== Résultats : CatBoost_Baseline ===
MAE   : 12.130 minutes
RMSE  : 18.919 minutes
R²    : 0.5657
Best iteration : 5998

🏃 View run CatBoost_Baseline at: https://patjedhahf-mlflow-ppml.hf.space/#/experiments/789482705024277871/runs/c1b65e0da4b843d280e9f7cd4e1ac101
🧪 View experiment at: https://patjedhahf-mlflow-ppml.hf.space/#/experiments/789482705024277871


In [ ]:
model2 = train_catboost(
    df=df_fe,
    run_name="test4",
    iterations=8000,
    learning_rate=0.055,
    depth=9
)



=== Début du run : test2 ===



2026/04/10 00:28:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Target Encoding terminé sur airline_name, dest_icao_clean et aircraft_model


2026/04/10 00:28:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/10 00:28:34 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!



Entraînement du modèle en cours...

0:	learn: 28.2814658	test: 28.2927481	best: 28.2927481 (0)	total: 37.8ms	remaining: 5m 2s
200:	learn: 22.2469563	test: 22.5381613	best: 22.5381613 (200)	total: 7.8s	remaining: 5m 2s
400:	learn: 21.4704365	test: 21.9807334	best: 21.9807334 (400)	total: 15.7s	remaining: 4m 57s
600:	learn: 20.8245701	test: 21.5400110	best: 21.5400110 (600)	total: 23.3s	remaining: 4m 46s
800:	learn: 20.3127763	test: 21.2050816	best: 21.2050816 (800)	total: 30.9s	remaining: 4m 37s
1000:	learn: 19.8537542	test: 20.9204473	best: 20.9204473 (1000)	total: 39.2s	remaining: 4m 34s
1200:	learn: 19.4412388	test: 20.6760140	best: 20.6760140 (1200)	total: 47.5s	remaining: 4m 29s
1400:	learn: 19.0577382	test: 20.4464181	best: 20.4464181 (1400)	total: 55.4s	remaining: 4m 21s
1600:	learn: 18.7146104	test: 20.2381573	best: 20.2381573 (1600)	total: 1m 3s	remaining: 4m 13s
1800:	learn: 18.4234665	test: 20.0773380	best: 20.0773380 (1800)	total: 1m 11s	remaining: 4m 5s
2000:	learn: 18.116

2026/04/10 00:34:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



=== Résultats : test2 ===
MAE   : 11.192 minutes
RMSE  : 17.855 minutes
R²    : 0.6132
Best iteration : 7999

🏃 View run test2 at: https://patjedhahf-mlflow-ppml.hf.space/#/experiments/789482705024277871/runs/6421a593892a41ef83909d76aef07ac4
🧪 View experiment at: https://patjedhahf-mlflow-ppml.hf.space/#/experiments/789482705024277871


In [162]:
model3 = train_catboost(
    df=df_fe,
    run_name="test5",
    iterations=10000,
    learning_rate=0.055,
    depth=9
)


=== Début du run : test5 ===



2026/04/10 01:04:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Target Encoding terminé sur airline_name, dest_icao_clean et aircraft_model


2026/04/10 01:04:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/10 01:04:03 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!



Entraînement du modèle en cours...

0:	learn: 28.2814658	test: 28.2927481	best: 28.2927481 (0)	total: 37.3ms	remaining: 6m 13s
200:	learn: 22.2469465	test: 22.5381613	best: 22.5381613 (200)	total: 7.47s	remaining: 6m 4s
400:	learn: 21.4752304	test: 21.9858594	best: 21.9858594 (400)	total: 14.7s	remaining: 5m 51s
600:	learn: 20.8525476	test: 21.5556720	best: 21.5556720 (600)	total: 22s	remaining: 5m 43s
800:	learn: 20.3072034	test: 21.1856324	best: 21.1856324 (800)	total: 29.4s	remaining: 5m 37s
1000:	learn: 19.8578299	test: 20.9057198	best: 20.9057198 (1000)	total: 36.9s	remaining: 5m 31s
1200:	learn: 19.4320824	test: 20.6440266	best: 20.6440004 (1199)	total: 44.4s	remaining: 5m 25s
1400:	learn: 19.0577424	test: 20.4161444	best: 20.4161444 (1400)	total: 51.9s	remaining: 5m 18s
1600:	learn: 18.7277031	test: 20.2167572	best: 20.2167572 (1600)	total: 59.5s	remaining: 5m 12s
1800:	learn: 18.3941945	test: 20.0199236	best: 20.0199236 (1800)	total: 1m 7s	remaining: 5m 5s
2000:	learn: 18.0837

2026/04/10 01:11:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'CatBoost_Delay_Prediction'.
2026/04/10 01:13:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: CatBoost_Delay_Prediction, version 1
Created version '1' of model 'CatBoost_Delay_Prediction'.


✅ Modèle enregistré dans le Model Registry : CatBoost_Delay_Prediction
   Version : 1

=== Résultats : test5 ===
MAE   : 10.903 minutes
RMSE  : 17.563 minutes
R²    : 0.6258
Best iteration : 9999

🏃 View run test5 at: https://patjedhahf-mlflow-ppml.hf.space/#/experiments/789482705024277871/runs/087a6fdb097145b5a507af6bcf0c3ca9
🧪 View experiment at: https://patjedhahf-mlflow-ppml.hf.space/#/experiments/789482705024277871


In [ ]:
# mlflow.end_run()

In [ ]:
# # Set your variables for your environment
# EXPERIMENT_NAME="ppml-mlflow-experiment"

# # Set tracking URI to your Hugging Face application
# mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])

# # Set experiment's info
# mlflow.set_experiment(EXPERIMENT_NAME)

# # Get our experiment info
# experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)


# with mlflow.start_run(experiment_id = experiment.experiment_id):

#     # Instanciate and fit the model
#     lr = LogisticRegression()
#     lr.fit(X_train, y_train)

#     # Store metrics
#     predicted_qualities = lr.predict(X_test)
#     accuracy = lr.score(X_test, y_test)

#     # Print results
#     print("LogisticRegression model")
#     print("Accuracy: {}".format(accuracy))


### Ajout des métriques

In [ ]:
# mlflow.log_metric("METRIC_NAME", metric)

In [ ]:
# # Set your variables for your environment
# EXPERIMENT_NAME="my-first-mlflow-experiment"

# # Set tracking URI to your Hugging Face application
# mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])

# # Set experiment's info
# mlflow.set_experiment(EXPERIMENT_NAME)

# # Get our experiment info
# experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# # Call mlflow autolog
# mlflow.sklearn.autolog()

# with mlflow.start_run(experiment_id = experiment.experiment_id):
#     # Instanciate and fit the model
#     lr = LogisticRegression()
#     lr.fit(X_train, y_train)

#     # Store metrics
#     predicted_qualities = lr.predict(X_test)
#     accuracy = lr.score(X_test, y_test)

#     # Print results
#     print("LogisticRegression model")
#     print("Accuracy: {}".format(accuracy))

#     # Log Metric
#     mlflow.log_metric("Accuracy", accuracy)


### On peut aussi logger les paramètres

In [ ]:
# mlflow.log_param("PARAM_NAME", param)

In [ ]:
# # Set your variables for your environment
# EXPERIMENT_NAME="my-first-mlflow-experiment"

# # Set tracking URI to your Hugging Face application
# mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])

# # Set experiment's info
# mlflow.set_experiment(EXPERIMENT_NAME)

# # Get our experiment info
# experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# # Call mlflow autolog
# mlflow.sklearn.autolog()

# with mlflow.start_run(experiment_id = experiment.experiment_id):
#     # Specified Parameters
#     c = 0.5

#     # Instanciate and fit the model
#     lr = LogisticRegression(C=c)
#     lr.fit(X_train.values, y_train.values)

#     # Store metrics
#     predicted_qualities = lr.predict(X_test.values)
#     accuracy = lr.score(X_test.values, y_test.values)

#     # Print results
#     print("LogisticRegression model")
#     print("Accuracy: {}".format(accuracy))

#     # Log Metric
#     mlflow.log_metric("Accuracy", accuracy)

#     # Log Param
#     mlflow.log_param("C", c)


### Logger le modele MLflow

In [ ]:
# # Set your variables for your environment
# EXPERIMENT_NAME="my-first-mlflow-experiment"

# # Set tracking URI to your Hugging Face application
# mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])

# # Set experiment's info
# mlflow.set_experiment(EXPERIMENT_NAME)

# # Get our experiment info
# experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# # Call mlflow autolog
# mlflow.sklearn.autolog()

# with mlflow.start_run(experiment_id = experiment.experiment_id):
#     # Specified Parameters
#     c = 0.5

#     # Instanciate and fit the model
#     lr = LogisticRegression(C=c)
#     lr.fit(X_train.values, y_train.values)

#     # Store metrics
#     predicted_qualities = lr.predict(X_test.values)
#     accuracy = lr.score(X_test.values, y_test.values)

#     # Print results
#     print("LogisticRegression model")
#     print("Accuracy: {}".format(accuracy))

#     # Log Metric
#     mlflow.log_metric("Accuracy", accuracy)

#     # Log Param
#     mlflow.log_param("C", c)

#     # Log model
#     mlflow.sklearn.log_model(lr, "model")


### Autologs

In [ ]:
# # Set your variables for your environment
# EXPERIMENT_NAME="my-first-mlflow-experiment"

# # Set tracking URI to your Hugging Face application
# mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])

# # Set experiment's info
# mlflow.set_experiment(EXPERIMENT_NAME)

# # Get our experiment info
# experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# # Call mlflow autolog
# mlflow.sklearn.autolog()

# with mlflow.start_run(experiment_id = experiment.experiment_id):
#     # Specified Parameters
#     c = 0.5

#     # Instanciate and fit the model
#     lr = LogisticRegression(C=c)
#     lr.fit(X_train.values, y_train.values)

#     # Store metrics
#     predicted_qualities = lr.predict(X_test.values)
#     accuracy = lr.score(X_test.values, y_test.values)

#     # Print results
#     print("LogisticRegression model")
#     print("Accuracy: {}".format(accuracy))


In [ ]:
# import argparse
# import time
# import os

# import mlflow
# import pandas as pd
# from dotenv import load_dotenv
# from mlflow import MlflowClient
# from mlflow.models import infer_signature
# from sklearn.datasets import load_iris
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score
# from sklearn.model_selection import train_test_split
# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import StandardScaler


# load_dotenv()

# # Tracking server
# mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])


# if __name__ == "__main__":
#     parser = argparse.ArgumentParser()
#     parser.add_argument("--n_estimators", type=int, default=100)
#     parser.add_argument("--min_samples_split", type=int, default=2)
#     parser.add_argument("--test_size", type=float, default=0.2)
#     parser.add_argument("--random_state", type=int, default=42)
#     args = parser.parse_args()

#     experiment_name = "iris_classifier"
#     registered_model_name = "iris_classifier"
#     alias_name = "challenger"

#     mlflow.set_experiment(experiment_name)
#     client = MlflowClient()

#     print("Training model...")
#     start_time = time.time()

#     # Keep autolog basic for a deployment/demo course, but log the model manually
#     mlflow.sklearn.autolog(log_models=False)

#     # ------------------------------------------------------------------
#     # Dataset: Iris
#     # ------------------------------------------------------------------
#     iris = load_iris(as_frame=True)
#     X = iris.data
#     y = iris.target

#     X_train, X_test, y_train, y_test = train_test_split(
#         X,
#         y,
#         test_size=args.test_size,
#         random_state=args.random_state,
#         stratify=y,
#     )

#     # ------------------------------------------------------------------
#     # Model pipeline
#     # ------------------------------------------------------------------
#     model = Pipeline(
#         steps=[
#             ("scaler", StandardScaler()),
#             (
#                 "classifier",
#                 RandomForestClassifier(
#                     n_estimators=args.n_estimators,
#                     min_samples_split=args.min_samples_split,
#                     random_state=args.random_state,
#                 ),
#             ),
#         ]
#     )

#     with mlflow.start_run() as run:
#         model.fit(X_train, y_train)

#         predictions = model.predict(X_test)
#         accuracy = accuracy_score(y_test, predictions)

#         mlflow.log_metric("test_accuracy", accuracy)
#         mlflow.log_param("dataset", "iris")

#         signature = infer_signature(X_train, predictions)
#         input_example = X_train.head(5)

#         # MLflow 3.x: prefer `name=` instead of deprecated `artifact_path=`
#         model_info = mlflow.sklearn.log_model(
#             sk_model=model,
#             name="model",
#             registered_model_name=registered_model_name,
#             signature=signature,
#             input_example=input_example,
#         )

#         model_version = model_info.registered_model_version
#         print(f"[INFO] Model logged as version {model_version}")

#         client.set_registered_model_alias(
#             name=registered_model_name,
#             alias=alias_name,
#             version=model_version,
#         )
#         print(
#             f"[INFO] Alias '{alias_name}' now points to version {model_version}"
#         )

#         # Optional: handy tags for the registry/UI
#         client.set_model_version_tag(
#             name=registered_model_name,
#             version=model_version,
#             key="dataset",
#             value="iris",
#         )
#         client.set_model_version_tag(
#             name=registered_model_name,
#             version=model_version,
#             key="metric:test_accuracy",
#             value=f"{accuracy:.4f}",
#         )

#         print(f"[INFO] Run ID: {run.info.run_id}")
#         print(f"[INFO] Test accuracy: {accuracy:.4f}")

#     print("...Done!")
#     print(f"--- Total training time: {time.time() - start_time:.2f} seconds")


In [ ]:
# REGISTERED_MODEL_NAME = "iris_classifier"
# MODEL_ALIAS = "challenger"

In [ ]:
# mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])

In [ ]:
# MODEL_URI = f"models:/{REGISTERED_MODEL_NAME}@{MODEL_ALIAS}"

In [ ]:
# MODEL = mlflow.sklearn.load_model(MODEL_URI)

In [ ]:
# MODEL

In [ ]:
# MODEL.predict(iris.data)